In [ ]:
!pip install -q transformers datasets evaluate accelerate peft bitsandbytes safetensors --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 13.3 MB/s eta 0:00:00


In [ ]:
import transformers
print(transformers.__version__)

4.57.1


In [ ]:
import os
from pathlib import Path
from datasets import load_dataset

In [ ]:
import torch
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [ ]:
dataSet = load_dataset("yelp_polarity")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38000 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased")

def tokenize(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

encodedTraining = dataSet["train"].map(tokenize, batched = True)
encodedEvaluation = dataSet["test"].map(tokenize, batched = True)

encodedTraining.set_format("torch", columns=["input_ids", "attention_mask", "label"])
encodedEvaluation.set_format("torch", columns=["input_ids", "attention_mask", "label"])

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/560000 [00:00<?, ? examples/s]

Map:   0%|          | 0/38000 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-cased", num_labels=2)

model.save_pretrained("./modelBeforeFineTuning")

sizeBeforeTraining = sum(os.path.getsize(os.path.join("./modelBeforeFineTuning", f)) for f in os.listdir("./modelBeforeFineTuning"))

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
model.to(device)
torch.cuda.empty_cache()
memoryBeforeFineTuning = torch.cuda.memory_allocated()

In [ ]:
from transformers import TrainingArguments, Trainer
import time

training_args = TrainingArguments(
    eval_strategy="epoch",
    learning_rate=2e-5,
    save_strategy="no",
    per_device_train_batch_size=16,
    num_train_epochs=2,
    fp16=True,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encodedTraining,
    eval_dataset=encodedEvaluation,
)

startTime = time.time()
trainer.train()
endTime = time.time()

Epoch,Training Loss,Validation Loss
1,0.152100,0.160477
2,0.111500,0.163406


In [ ]:
model.save_pretrained("./modelAfterFineTuning")
sizeAfterFineTuning = sum(os.path.getsize(os.path.join("./modelAfterFineTuning", f)) for f in os.listdir("./modelAfterFineTuning"))

In [ ]:
from sklearn.metrics import accuracy_score
def accuracyFunction(model, dataset, batch_size=32):
    model.eval()
    preds, labels = [], []
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size)
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            preds.extend(outputs.logits.argmax(dim=1).cpu().numpy())
            labels.extend(batch["label"].cpu().numpy())
    return accuracy_score(labels, preds)

In [ ]:
print("GPU memory before Fine tuning: ", memoryBeforeFineTuning, "Bytes")
memoryAfterFineTuning = torch.cuda.memory_allocated()
print("GPU memory after Fine-Tuning: ", memoryAfterFineTuning)
print("GPU Usage", memoryAfterFineTuning - memoryBeforeFineTuning, "Bytes")

GPU memory before Fine tuning:  263661056 Bytes
GPU memory after Fine-Tuning:  813573632
GPU Usage 549912576 Bytes


In [ ]:
print(f"Training start time: {startTime/60:.2f} minutes")
print(f"Training end time: {endTime/60:.2f} minutes")
print(f"Training time: {(endTime - startTime)/60:.2f} minutes")

Training start time: 29385141.43 minutes
Training end time: 29385216.41 minutes
Training time: 74.98 minutes


In [ ]:
print("Model isze before Fine-Tuning is: ", sizeBeforeTraining, "bytes")
print("Model isze after Fine-Tuning is: ", sizeAfterFineTuning, "bytes")
print("Model size-difference:", sizeAfterFineTuning - sizeBeforeTraining, "Bytes")

Model isze before Fine-Tuning is:  263145217 bytes
Model isze after Fine-Tuning is:  263145266 bytes
Model size-difference: 49 Bytes


In [ ]:
accuracyBeforeFineTuning = accuracyFunction(
    AutoModelForSequenceClassification.from_pretrained("./modelBeforeFineTuning").to(device),
    encodedEvaluation,
    batch_size=32
)

accuracyAfterFineTuning = accuracyFunction(
    AutoModelForSequenceClassification.from_pretrained("./modelAfterFineTuning").to(device),
    encodedEvaluation,
    batch_size=32
)

In [ ]:
print(f"Accuracy before Fine-Tuning: {accuracyBeforeFineTuning}")
print(f"Accuracy after Fine-Tuning: {accuracyAfterFineTuning}")

Accuracy before Fine-Tuning: 0.500421052631579
Accuracy after Fine-Tuning: 0.9523684210526315
